# Granite census — untouched tiles only (Kaggle)

Layers **20–39**, neurons **0–8191**, probe `Who are you?`, dose −4.0,
`max_new=50`, batched at **32** per forward pass.

**159,652 tiles in this range have never been steered on.** The
4,188 that have are excluded by an embedded skip-list, so no budget is
spent on a tile we already own.

Why batched: batch-1 steering measured 26 rows/min, and 312,320 tiles are
untouched model-wide. Batching is the only way the census fits in a week.

Two fields keep the rows honest:
- `batch_size` / `slot` — batching can change a greedy continuation, so these must
  not be pooled silently with batch-1 rows.
- `dtype`, `device` — P100 is fp16, the laptop is bf16.

Budget 20000s: the session stops cleanly instead of being killed mid-row.


In [ ]:
!pip install -q -U 'transformers>=5.0' 2>&1 | tail -2
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('gpu count:', torch.cuda.device_count() if torch.cuda.is_available() else 0)
print('transformers', transformers.__version__)


In [ ]:
%%writefile batch_screen.py
#!/usr/bin/env python3
"""Screen many tiles per forward pass.

Why
---
Batch-1 steering measured 26 rows/min on a P100. That number is not a model limit,
it is a utilization limit: a 3B model emitting 50 tokens is bandwidth-bound and one
sequence leaves most of the GPU idle. A screen needs only one answer per tile, so
the way to cover 300k tiles is to put many tiles in the same forward pass.

All sequences in a batch must share a **layer**, because the hook lands on one
block. Within a layer any mix of neuron and dose is fine: each row carries its own
offset vector, so an answer never depends on which batch slot it got.

Every row records `batch_size` and `slot`, because batching can change a greedy
continuation (the Qwen reproduction documents this). Rows written here should not
be pooled silently with batch-1 rows.

    python batch_screen.py --layer 12 --neurons 1000-1100 --probe "Who are you?"
    python batch_screen.py --layers 0-19 --neurons 0-8192 --out screen_L0_19.jsonl
"""
import argparse
import json
import os
import sys
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = os.environ.get("GRANITE_MODEL", "ibm-granite/granite-4.1-3b")
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")

DOWN_PROJ_PATHS = ("down_proj", "output_linear", "c_proj", "w2")


def mlp_of(model, layer):
    blk = model.model.layers[layer]
    for path in DOWN_PROJ_PATHS:
        obj = blk.mlp
        for part in path.split("."):
            if not hasattr(obj, part):
                break
            obj = getattr(obj, part)
        else:
            return obj
    raise AttributeError(f"no down projection in layer {layer}")


def parse_range(s):
    if "-" in s:
        a, b = s.split("-", 1)
        return int(a), int(b)
    return int(s), int(s) + 1


ap = argparse.ArgumentParser()
ap.add_argument("--layers", default="", help="e.g. 0-19; overrides --layer")
ap.add_argument("--layer", type=int, default=12)
ap.add_argument("--neurons", default="0-8192", help="START-END within each layer")
ap.add_argument("--probe", default="Who are you?")
ap.add_argument("--dose", type=float, default=4.0, help="magnitude, applied negative")
ap.add_argument("--batch", type=int, default=32, help="sequences per forward pass")
ap.add_argument("--max-new", type=int, default=50)
ap.add_argument("--max-seconds", type=float, default=0.0)
ap.add_argument("--dtype", choices=["float16", "bfloat16"], default="float16")
ap.add_argument("--out", default="")
ap.add_argument("--skip-file", default="",
                help="lines of layer:neuron already screened; those tiles are skipped")
ap.add_argument("--resume", action="store_true")
ap.add_argument("--dry-run", action="store_true")
a = ap.parse_args()
if not a.dry_run and not a.out:
    ap.error("--out is required unless --dry-run")

# --layers is INCLUSIVE (0-19 means twenty layers); --neurons is half-open
# (0-8192 means 0..8191). Mixing the two conventions silently dropped L19.
if a.layers:
    if "-" in a.layers:
        lo, hi = (int(x) for x in a.layers.split("-", 1))
    else:
        lo = hi = int(a.layers)
    LAYERS = list(range(lo, hi + 1))
else:
    LAYERS = [a.layer]
NLO, NHI = parse_range(a.neurons)
DT = getattr(torch, a.dtype)

skip = set()
if a.skip_file and os.path.exists(a.skip_file):
    for line in open(a.skip_file, errors="ignore"):
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if ":" in line:
            lp, np_ = line.split(":", 1)
        else:
            parts = line.split()
            if len(parts) < 2:
                continue
            lp, np_ = parts[0], parts[1]
        try:
            skip.add((int(lp), int(np_)))
        except ValueError:
            continue
    print(f"skip: {len(skip):,} tiles already hit, excluded", flush=True)

if a.dry_run:
    live = sum(1 for L in LAYERS for n in range(NLO, NHI) if (L, n) not in skip)
    print(f"device={DEVICE} layers={LAYERS[0]}..{LAYERS[-1]} neurons={NLO}..{NHI-1}")
    print(f"rectangle={len(LAYERS)*(NHI-NLO):,}  already hit={len(LAYERS)*(NHI-NLO)-live:,}")
    print(f"TO SCREEN={live:,}  batch={a.batch} max_new={a.max_new}")
    sys.exit(0)

done = set()
if a.resume and os.path.exists(a.out):
    for line in open(a.out, errors="ignore"):
        try:
            r = json.loads(line)
        except ValueError:
            continue
        done.add((r.get("layer"), r.get("neuron")))
    print(f"resume: {len(done):,} tiles already present", flush=True)

tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
m = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DT).to(DEVICE).eval()
n_layers = m.config.num_hidden_layers

calib = ("Humans are people. A person has feelings, friends, family and a home. " * 20)
if os.path.exists("corpus.txt"):
    calib = open("corpus.txt", encoding="utf-8", errors="ignore").read()[:4000]
cids = tok(calib, return_tensors="pt")["input_ids"].to(DEVICE)

_med, _norm = {}, {}


def calibrate(layer):
    """Median residual norm entering this block, and per-neuron column norms."""
    if layer in _med:
        return
    at = min(layer, n_layers - 1)
    with torch.no_grad():
        hs = m(cids, output_hidden_states=True, use_cache=False).hidden_states[at]
    # token 0 excluded: it is BOS in chat templates and inflates the median.
    _med[layer] = hs[0, 1:].float().norm(dim=-1).median().item()
    W = mlp_of(m, layer).weight.float()
    _norm[layer] = W[:, NLO:NHI].norm(dim=0).cpu()
    del W
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"[calib] L{layer} median_resid={_med[layer]:.2f}", flush=True)


def run_layer(layer, ids):
    calibrate(layer)
    med = _med[layer]
    norms = _norm[layer]
    dim = mlp_of(m, layer).weight.shape[1]
    t0 = time.time()
    made = 0
    for i in range(0, len(ids), a.batch):
        chunk = ids[i:i + a.batch]
        if a.max_seconds and time.time() - t0 > a.max_seconds:
            print(f"[budget] stopping L{layer} at {made} rows", flush=True)
            return made, True
        texts = [a.probe] * len(chunk)
        enc = tok(texts, return_tensors="pt", padding=True).to(DEVICE)
        B = len(chunk)
        off = torch.zeros(B, dim, dtype=DT, device=DEVICE)
        for j, nid in enumerate(chunk):
            off[j, nid] = -a.dose * med / float(norms[nid - NLO])

        def hook(mod, args, off=off):
            return (args[0] + off.unsqueeze(1),) + tuple(args[1:])

        h = mlp_of(m, layer).register_forward_pre_hook(hook)
        try:
            with torch.no_grad():
                out = m.generate(**enc, max_new_tokens=a.max_new, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        finally:
            h.remove()

        width = enc["input_ids"].shape[1]
        for j, nid in enumerate(chunk):
            ans = tok.decode(out[j][width:], skip_special_tokens=True).strip()
            rec = {"layer": layer, "neuron": nid, "probe": a.probe,
                   "dose": -a.dose, "median": med, "answer": ans,
                   "tile": f"{layer}:{nid}", "batch_size": B, "slot": j,
                   "max_new": a.max_new, "dtype": a.dtype, "device": DEVICE,
                   "screen": True}
            with open(a.out, "a") as f:
                f.write(json.dumps(rec) + "\n")
            made += 1
        if DEVICE == "cuda":
            del out, enc, off
            torch.cuda.empty_cache()
        el = max(time.time() - t0, 1e-6)
        rate = made / el * 60
        print(f"  L{layer} {i+len(chunk)}/{len(ids)}  {made} rows  {rate:.0f}/min", flush=True)
    return made, False


total = 0
start = time.time()
stop = False
for layer in LAYERS:
    ids = [n for n in range(NLO, NHI)
           if (layer, n) not in done and (layer, n) not in skip]
    if not ids:
        print(f"L{layer}: nothing to do", flush=True)
        continue
    print(f"=== L{layer}: {len(ids)} tiles", flush=True)
    n, stop = run_layer(layer, ids)
    total += n
    if stop:
        break
    if a.max_seconds and time.time() - start > a.max_seconds:
        print("[budget] session budget used", flush=True)
        break

el = max(time.time() - start, 1e-6)
print(f"\nDONE {total} rows in {el/60:.1f} min = {total/el*60:.0f} rows/min", flush=True)
print(f"wrote {a.out}", flush=True)


In [ ]:
import base64, zlib
BLOB = 'eJw9nVGC7SgIRP9nNVcKRXv/CxulTt5P6ORKYkyBCEjH7+/3X/z+zjuM6GP/HfWOK9+x+nq5zW+Y7CZjmpT5fya+KN9umpSblPn26ocMmfTFCF+cPqu+WRxImPRv+nVfNJaJL/rpim4pDRPOZNKdUKaJ2f0glS9utzz9DunXzOqz+fuZ9NPn8JmHYEbzTfm39Nnyb8sXyxe3+XZfXL806b6s4TOP4Ep/hOmzyZn51jAxnzu/3Pm1zXD6rPxZS32xfM/KvkvN7kstt+RzLzep5c/eD9o/iEyaffvW273e/gA7OGv27e+w3fntb7v9Ubc7eHKYhEmadMvjDh6/9Fk+W/7NL33cz+NhvfgUNKEbCn6Nj0u5vjj3SF5Kuw2/kTeGP9QYguYPyrlHbQDtMQzVKxPuTyAqQT/Cw3Qp14UMid+n7xOL3+ujtNsB5dzf6VLzaQwo5wH1MA/5iwz5ow9xH8EP9EdaE4xEnNPfYGQNKL8zXsn7J+M2jeyBnFx14PebjN+kP5NxnHNAacfzkKUxi/aMB3I05oGf/q8Y0IBy3apgLPTSmuZb66O04/4I06Uej3VQZD/0Gt8PiRoFnsogvXqP6/SzGBck6dIF9e/bauVSrjP+m3HaVi3jMI6H/h8rhEv5HTyezbll7tLv3GoWOYifx+NSQSfU8wGKP5CDGMF5oMPdz0tprx+UdqKdaLd9PX4of2uPS90uJud+v0vNF5vrfo+QlfKlvo8sN5dyvmhnvAZ6/lJ+534yfiP9XSItP5HMSpnQyfX1Ua5bbwRycSn3sXxcyu88ZxqXgRxcaj7wfymzIf2cTITT+Iv1gw4o32NZr8SS+Rf3XdwPnMdifl3WG3e6TSjTL/cB35diF/A9S5wn7Rifov+1PsrvPK/2R93vzXO39UOA79iM595cB6+H/hzwdsDV4X0P+GJWCaaVS2lHf09yH/p98uPjOZbnYAa6lPshJwc5Qf4u5X6MM/J4qS2Q329CMWH83fTDQvm5v5dyfXLu6e3Sz/TB6LE8CnkUcibsrUttyGBxXSPJ5/HDTIIv6BfWl0Jc9/vfaYP21idinrnUfPp99tYP6nYKfp+cW0702Vk6H8Umo18ZH3X7tN1ybTbo+ih8m3YHPu476cdkfJGnqx64Xly3fIp54lJB3X7x3GVcCDkS5th9LY/TstwLU0zI1aX+vRjnYpyK5yEvl/I7963PHo2P+rmb77v5vnt91Hyb77zByflh0PK9j/WGDsbz4f7IlZAbIS9CLoR9dSnteb/j59zPF1CbzD9hQfs++bMcJbi+lHYFn7/DNTto7+ckS4xkVZHYX5f6uSO5bnm7lOuerzMsFxk8N/g9/N2Sdcalvm/w3DAuU5azSzkPzj1vJmuKlPV4yvKestxm/qDxUfMxn1zK+fyo+5Ufv+28ZN7I9LyX4Dyxq3Jy/8l9Ju8zN7/b7sj58RmX15zz72twbjwkeL+U68ZbYi/daS+gfv9inIr+1GDpZHlK5olLuV6087yYrDMudTvsody255OFxKXm23y3zbhs64FLff9j++PSgPp38J3gOllTJPZSHnCI3ZSHlSD20l0DpqnXPpN1xmR9MX+Tc9Z+P5Z74PrSvv9dPUKtBya4nsP9m2Nx3e81x+H6Yb3p8b7U7SI49zx916N+XnjcL/V9kINL4Zvcr/idfobxMWV8TNYRk9X2ZIE9xftqcb04L/h5X9YT18zxc9M4m0k/sJ8mdtNM28vzW2Z/6+xJPybjPcV1Vt4zObccTNYPlyaU3xf34bmsJyb21mX3+yEn9zYs6S2/93a+L+vxCe4v5ZzvWrTHbrp0QblOf7GfJuuIydJ8Yj9NluOXFhR+8FXgpIz/uRnvPT7q+yI/k3lkbnC2uf/enG/agTfmg4l9c5dZvt+hHwc+5ofJ/HAp1623FnKykJNrxkKtbxd2zRq0w65Z6Ps1bM+t4Xlvjc2533shJyvwqIT1yDWPOfd73M/oc9bJC3tlYacs1gNL/v7XjMYnY/vkUt8n6V/inmF9fKn7g11y6Xfu++FFWtgpCzvlUj9/Wk+uOblu/bvA9WJ9fM0QqPG60O8LvK7F+7FOuGobynOWOOd56Pf1eZ2W5+f1+ZbwKl3q31kHL9YJi3XCKuufBc4X64WFfbOKccLOWcX9i/FiHrjU/Nv24drBuThPXF84y7b129rcH3wv7KSFnbQ24wbeF/PGOjzngMPj+Wcdy/PCTlqsOxZ2/6Xwc/9T/A7ODu/DfFI/4/MOn11wv8n54tzjcpdjAe37F/Z+MX/UMK4LH2yx7r6U9sm55eaqKdrj8xs4/Yb1YmE/XQq/7dpCzi615zC4L+uECrzEeGpL9FO8J/6nwiNbrAuK9XnhhyoV58bzhc0P+p1PKI5M67lKfdT9Sct5Mb9c+p37OazHi/XDpeaflptrlg9oQN2Odfulvs/URyfU/UJ+r1nP9YnHdXFf3ndunse4LuPuLp9xzSbnjDPrjMI/VYtxWp5ni/VGFXhALov5pz7PL3JY/7y9fD/ml2JeueaZ22/wxrxSyN+l5tvgZXOfbbwX88cVa9+HdcelG8p18HGC6/SX9Xwx/xTr8UK+Ci9wnc9dDU6P8bHxB2/kbOPv2j9xXZxPfl84uq037mvb1f0zTjb22zVPPxf4NOV+Q1xPfOO4vJG3Pdzf3fPUcNRiOEQw7P8f9v8PO/6HvfPDjvhhn/uwY3zYFT7sAx84vwfO7YFzeuCUHjiNB07jgZN34OQdOHEHTtyBc3bgZB04VwfO0IHzcuCcHDglB07GgXNx4EQcOP0GTrmB823gZBs4zwZOrYEza+A0Gjh1Bk6dgVNn4NQZOHUGTp2BU2fg1Bk4dQZOnYFTZ+DUGTh1Bk6dgVNn4NQZOHUGzpiBM2Xg3Bg4NwbOh4FzYeBcGDgNBs6AwSJ7sAgbLKoGi6bBommwqBgsJgZG+8A4HxjPAyN5YIQOjMyBMTYwsgZG1sBIGRglA2NkYGQMjITB5D6YXAeT6GASHUyCg8lpoGwGSmAgHAPQh0EfBn0Y9GHQh0EfBn0Y9GHQh0EfBn0Y9AHoA9AHoA9AH4A+AH0A+gD0AegD0AegD0AfgD4AfQD6APQB6APQB6APQB+APgB9APoA9AHoA9AHoA9AH4A+AH0A+gD0AegD0AegD0AfgD4AfQD6APQB6APQB6APQB+APgB9APoA9AHoA9AHoA9AH4A+AH0A+gD0AegD0AegD0AfgD4AfQD6APQB6APQB6APQB+APgB9APoA9AHoA9AHoA9AH4A+AH0A+gD0AegD0AegD0AfgF4GvQx6GfQy6GXQy6CXQS+DXga9DHoZ9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AvQC9AL0AtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1ALUAtQC1PmSR/Kv/4p3OH3af77+ZyeT3N9WH+sd1SxqntktXw/z7+EkO+HkHpur+tfq+1Rf2c21ff/fMGnuweNGuQfuV/iiH9/5KI/4op/dq55HzLd8s/JZ+Wy7935e/MrELza6SfCC4TP5LH8mPptm98uGH9u+50fCxBe3mxz/dvzbSZPtMTwexDBZJv7N795pMI940D3e8hB0Nkw6GyadDfPINPHN3E/5q8ifRe613OuWkEfccrvldhO/Q6/V8vn2TfpiRnc+DYTM7lmvyvKvg6mXlEzM7iFox3c6CSedhJNOwkkn4Tw8ASu39BOmX7P9g+nMnEf6eXP6N7/YNO6mnz5rmbjJ9s22m7hL7RtM5/Ckc3jSmv2R5lvG4PIHWDLWE7C7yXSTafbJRd96mc/jsgzTZZguf4DlkV8e+eWRX+7g8pgtw6YM4fpxZikziMpiUemz5KxM3HL+TMxuaJShUR7BcgfLQ1fWEfsHkck2QaJ/JmHSN9uGcEe+0olE6USiR6aJ72IZ2x6z7Z61x+cRP8/43P5w21K1PS7HnTiWnOPHHuPzGDbH+Dwel2PY9CSTnpkfcRN/sY4nPGJ2f6pjCT/+YsfDcwyps93SHTwW9GPJ6UXsU3Lu6NV5P6igCd1Q2gU6Eg34C35PlGbCn7Sf/L7gM96cDNVK9lO28G2eu+mf4TY+pTwsEWPER9HS4tzf7VJBuT65jnYek+v061PXn74e9G8U97GuHuNwbuA7Casp58xRwbQRjFvQ3/hmDjGfiN/zm1gWlOuL6/QPVe5kraa0YxyDOSzobzCOcfj9MGWNAf2mMCjjKPolvqvym+N8PxT3QEk7yStJ8spnFQrq3xNcJRNoevoYaeFzEtij3D9577bmkmSwxLpMrMtHD+3AybSqvNTt0d+XbqjfawbUqvzSAaUd32Py3qj1S+GbtKe/k+816TeKf0zwPov78P0m323y3VD/TjZLrOSm2A8/zAnea3lCchJakoSWJKE9akVyYU27f1aJ77/WR+HDQFmM9wL3qz4z5rNjON+0p3/trG8qqO9f4KzAWdHfQg6KfjITOMmtKfzIY30GFHJZjF/RnwIPxTgyQzgpLt8q46OCLqjbM12MDV42/d3gZdPfDW43uNjoGSaKsTf34btuxmmjLw7PP3xHJopxkI+DnclccdUNfHy3w3OOp28n5TXFKma8Dt/toFeZBJysl6yukmS9ZJX1qDBGLY+XYpVic/48fzupL0nqS5L6mnIfy0Og5wO9Hr/P5sX6/LkfgX4P9HtgdjsZMEkGTJIBk2TAJBkwSQZMkgGTZMCm7s/AWGYeCExwryaTZMFHrb8vdT/C+srJg48yLsF9YtLO3+Xa69xn8/umvXHq5MKm5pONTicZPmp5CfRyiPcX74m9fSntLTchVgGa8M1vxeDnqfidFYHqu04/GA/Rb/G99PXzcF/L+52uAiooSxP6n4xfsoBB3wfG+aWcJ+3BVfI+yfskOEvwlIvri/su7sO4J3hj3ggM/mD+cLLko7wn88n9nKyieJ9pfRvMF8ECwMmVSXJlklyZJFcmyZVN/bzJuM7N+eZ+LMLQ74Fej2U9EBj5l7Ks4/nLdkSg752MmSRjNnW/F3KJ7X8p9wEP6H8nazbld+RhIafLejTQ74FeD0z9wNZ3UmeS1JkkdSZJnU1ZqYvrSXvkshi/Wh/lefSTecDJnok3pynt+Y7FOBdr3U1/0euX+vfNeG76uZHrTf/Q65dy7nn9Uu4DzjbP3ftbYnN/vvNGz6H/4yDvh34d5IUlgpNQkyTUJAk1SUJNklAfBX8sH+zFSlyrTf185gknizaFn+99LNdOIm36neMPGPhWWPz/rAcv9RqdeULY/8L+v5TfWfz/kvtM2k3Oi+ex2GcdcCl89I/5QcwPwv4Xel/Y/cLuF/r+Uj9n8FzsfyextoeD3z1finlBzAtObm3K75YHe/Wa4g2hX8E4xqCd5VRBP0O0E9cZn0h8LMabsPsvFZT29Bf7XvH1x8s6J8029XPE9/ucNwraJe0YH+x6fZ4Y8TxZr+pzzXy+GfHdxPOZJ8Q8IeaJS82fjFfyHZP+MU9cuqC4juhv4m1KcMd84uTdptwf/CXvlZ/TaX4+KO6L+ynX55TiOt+fdYXS8ivWF2J94STgpubDaXRpQn3/yXtM3GSsL5w03BRvmO0mMW/c6Zvf6d+kf6wfhNNIzCOajC/rg0t9fQ3OB+eM42IcmT+0cNXhPxLzyKW0z89Px/3AKesIMb8I55LwLml9jr3FeXFf8MN8I+YXJzMnXu8kmbkpvkDwUbwP842YX1R4HplHLuV3xrfoX9G/Yjzr8zXy/ZlHxLpCzCNiHSBcR8J3JNYD2vq8lAH9zvndeli4kIQPycnVj9Kfzffdn9cTeWLdIOaNSyfU7Q/jc+gf84gO+gZvk5gnnHzd1O+Fx0nMG8KFpGP71cnZSXJ2kpydJGc3TahdsfiPLsVRO2iPj/aHk/Ynfhe/J/dzP53cnSR3N8XjO3nO5H6L84K/uF99HmLu5/dIvPvJOiPx6Cd+pMSZ72TwR/1dE7+Rk8Kbfue029wXB3UYxxm2NxJPfjIfJPNBBu8byTnvi1s/mQ+SecBJ5kmSeZJk3pTfeU/8P4nTP1l/JH4gJ6MnyehJMnpTfOmMD+sQJ6c/yvjg6neSelPfXxNqvCfzSLIOSeaPq74/yn3op3DNs65I9L6T3B8VvyfXk+t8p8/tj75P1g/5xQGyPsrvGz7GjZhAou8zrWdzDqIKjAvrAifJP8p3nTx3Ws6TCECix51E35TrPHdu7sv74w9KAgKJnr/U90e/J+uCJCqQrAsSfe1k+yTJPokiNhXU44+evssp2tEPPPzJOsDJ+EkyfhKNTKKRSTL+o/Ojfj72fOLPSfRu4s9J/DZO1m/q3/HbJPZ84rdxEn+SxJ8k8T9K9Ac/TeLBd3J/U78v9ruT/Jt+193+8N7o30TPJvrVmwGa0k5fSIl29AN9mwe5wJ+TOPUTuzzx4HuzQLJZoCnPQz5w4yd+/MSPc82RL5blMNTPePWmgmRTQdOCOuyEHT/x9zia3JRzAlnoWUeZk80HTReU9pt27qc3JSSbEpJNCU25ThBteLwuNT92+0T/zrEIyy3OCakNYmpEWCd2/KXcl37ix5/Y8XN8/TvfdfNh30+CsxM7fxKencRnJ/6hGV+A0PaAN0skmyWSzRJNiSdab038SBN9780TSfQ+2TyRbJ5oynXrA0f3m7o/RG4n/qNJ7Haityf+o4lf35ssHmU80dMTO9+bLR7d/L6/mKfb4f+Z+Pkn/h9vwuigqPnQyxO//sRPM/HPXEp73hf/zMQOn19U9gvLfnHZyXtib0/8NRN9fKn7NUV7cDb5DpPxxw6f6O2JH9+bO5LNHcnmjmRzR4d5v3Avv4M7/Ppzno+6/WK8sMcn9vhc4GbRX8K6E/vcm0Ka8vvkfovfrRcmfpmJ/TzR0xO7eeJ/n+jrS4lF81zs6UnUduK3cZbIo+CGOO7EfzPx309CuZfyHMYVf85E/0/0/8T+nvUvJl5QfgcH2OMTP//ELp/MDxN//iQIPPHfe5NKsknl0aA978l8cSntea/9ZZCsj3Ifvvvme2/6vZEP/EDe3HLp4bmH52B/T/w0E7+9N708WsT/uT9+mokdPrHDJ3a4N8Ekm2CaOpDPPLCwvxf290Lfr19wHpwLOr+MgwHdUH4v7kuKAPb1wn+ziOcu5gFnEyXZRE3JYbA8L+xvb8p5NGhPWgP2tzfnPEpKwyCnYZDUMMjywW/jzTtNv+vwbfjIZmAeWOh9b+5p+p27X8H44c+55lZAfX/s8IVfZpFrs7CzF379S30/fdkblrtFvHWJ7AxZ/yz8MAu9vNDLl/q98dMvMmG8majpd05eiOephR9lEXe9FP6EMq74SxZ280JvXwo//cLvvtDXCzt64TdZ2M3OHnvJJz9SUhhX9PlCfy/itGsyPvhLFv6SRc6NNzElm5iSTUxJVlqyiSnZxJRsYmrKue3LS/0c9PLCf+LNTcnmpmRz06OineCjX+jphX5exF0X9vb6UnGW5Xt9uTVfHs2iH9jZCzt74V9fJNMs/OsL//rC77Hwqy/8Hgt/x8L+9qaopvz+ZfqAp6I/xXfD/vYmqWSTVFP3j0wab5pKNk0lm6aaLmhBaed5wlmCyWaqZDNVspkqyR58tD4KH/jaXzYS/cJOX8RRL/U4nSBdKb5z9ws7feFfX/g/nKXYlHaMH/b5Qk97E1ayCSvZhJVswko2YSWbsB71/FnEUQv/+KXfdWdCEU/1pq1k01ZTfid3Ef1b6N3CDi/8G4X9Xejbwv72pq5kU1dT2gXX/R2LuGnhVy/0b+FPL/RvoXcLP0iRR1P4Q4rER2/2asrvm/t9/dzkjDF+6N8if6awr4v4a2FXF3kzhX19aUHhm7RnHMmIvBT+zfXzUZLWGDfxHfUlszFOJDwWcdjCji786EXctciTKeKt3nzWFP79Xacd/cCvXiQ6Fv70ws4u9HlhXxd+cG9SSzapJZvUkk1qySa1RxkP4qF3uc7vxrM3rXVm3oAKmtCCksE3aMc4YZcX8VJvaks2tSWb2pJNbU15HvIw6Sd2eaHPvckt2eT2KPif9B/9XsRXC/v7Un4nYXHxXsvzz6XuL3HWwp9S+MMv/c59//VlJ5KXiN+k0O9F3syltOe7E1f1prpkU92lX+pkWX95k12yya4pv3vevjSh8NM//C71ZVZ+qZXMB/VlVf7Lp6Tf+MOL+aCwwws7vLDDvWkv2bSXbNp71H6FIs5a2NlFfLWYH4o4azE/FPNDYW8XdnZhZ9emf/jHi/mg8I8X/htnkSebApuan/yawm9TZGQWdnkRdy3mBW8SfJT+4tcp4q6Xwod84dcpEjeLeaTI0ynmjyL+Wqf4HX1A4madL7HVcn6nW6eTMn9s8nO8uTDZXPio8bnxk2/ycjb5OBs7fpN/eSn39/jez0g78uDx42zs901+jjcldlat7zf+pddOqNuTf7Ox2zd+nY1fZzOP7JHwedw2eTmbuOxmfvEmx2STY7LJsSn3pb9tx8+3a2D+9V/xDnqHfIf5Dusd6h32O5xubJbmGc00mms022i+0YyjOUezjuaN5g0/r3mjeaN5o3mjeaN5o3mjedW8al65s82r5lXzqnnVvGpemdcPbh/EdFr4dD74dM73/Ov42iNusn8m/bhOsJ5Oop7OkJ5OOZ5OK55OJJ6kA0/SfCfptJN02Em66STNdJLuOUnvnKRrTtI1J+mUk3THSZrjJJ1wkpY3Sa+bpMFN0tUmaWqT9LJJ+tYkzWmSvjRJO5qkAU3SaSbpM5P0kUl6xCT9YBLWn4S/J+HuSTh3En6dhFMnYc9J2HISXpsvzDSgAf2u9/3sxp+44Sfu84n7fOJ+nbhZJ+7Aibtv4j6buMEm7p2Jm2biHpgsvyfL78lybbIcmyy7JsumyXJmssyYmOkTc3xihk/M5Il5OZ/50r97+p9MUxO1PFELEzFdT0zXX/8V76B3yHeY77Deod5hv8PpxmZpntFMo7lGs43mG804mnM062jeaN7w85o3mjeaN5o3mjeaN5o3mlfNq+aVO9u8al41r5pXzavmlXn94BbTZTFdFtNlMV0WxWVRXBbFZVFcFsVlUVyI4kIUF6K4EMWFKC5EcSGKC1FciOJCFBeiuBDFhSguRHEhigtRXIjiQhQXorgQxYUoLkRxIYoLUVyI4kIUF6K4EMXFvrxFRtMio2mxL2+xL2+xL2+xL2+xL2+xL2+xL2+xL2+xL2+xGXWxGXWxGXWRMbVQBQtVsFAFC1WwUAULVbBQBQtVsFAFC1WwUAULVbBQBQtVsFAFC1WwUAULVbBQBQtVsFAFC1WwUAULVbBQBQtVsFAFC1WwUAULVfBw6PFb4fGzaliohoVqWKiGhWpYqIaFaliohoVqWKiGhWpYqIKFKliogoUqKAtJWUjKQlIWkrKQlIWkLCRlISkLSVlIykJSCEkhJIWQFEJSCEkhJIWQFEJSCEkhJIWQFEJSCEkhJIWQFEJSCEkhJIWQFEJSCEkhJIWQFEJSCEkhJIWQFOmXhbAUwlIISyEshbAUwlIISyEshbAUwlIISyEshbAUwlIISyEshbAUwlIISyEshbAUwlIISyEshbAUwlIISyEshTAU4e1CKAqhKISiEIpCKAqhKISiEIpCKAqhKISiEIpCKAqhKISgEIJCCAohKISgEILCrVW4tQqhKISiEIpCKAqhKISiEIpCKAqh2G9+3H/9V7yD3iHfYb7Deod6h/0OpxubpXlGM43mGs02mm8042jO0ayjeaN5w89r3mjeaN5o3mjeaN5o3mheNa+aV+5s86p51bxqXjWvmlfm9YNb9LdFf1v0t0V/W/S3RX9b9LdFf1v0t0V/I/ob0d+I/kb0N6K/Ef2N6G9EfyP6G9HfiP5G9DeivxH9jehvRH8j+hvR34j+RvQ3or8R/Y3ob0R/I/ob0d+I/kbkNyK/EfmNyG9EfiPyG5HfiPxG5DcivxH5jchvRH4j8huR34j8RuQ3Ir8R+Y3Ib0R+I/Ibkd+I/EbkNyK/EfmNyG9EfiPqG1HfiPpG1DeivhH1jahvRH0j6htR34j6RtQ3or4R9Y2ob0R9I+obUd+I+kbUN6K9Ee2NaG9EeyPaG9HeiPZGtDeifQz6Y9Afg/4Y9MegPwb9MeiPQX8M+mPQH4P+APoD6A+gP4D+APoD6A+gP4D+APoD6A+gP4D+APoD6A+gP4D+APoD6A+gP4D+APoD6A+gP4D+APoD6A+gP8x3B/AfwH8A/wH8B/AfwH8A/wH8B/AfwH8A/wH8B/AfwH8A/wH8B/AfwH8A/wH8B/AfwH8A/wH8B/AfwH8A/wH8B/Af5ruDEByE4CAEByE4CMFBCA5CcBCCgxAchOAgBAchOAjBQQgOQnAQgoMQHITgIAQHITjMd4f57iAUB6E4CMVBKA5CcRCKg1AchOJYKPT+U8g99F/xDnqHfIf5Dusd6h32O5xubJbmGc00mms022i+0YyjOUezjuaN5g0/r3mjeaN5o3mjeaN5o3mjedW8al65s82r5lXzqnnVvGpemdcPfqIv/0cO+T9yyP+KQ/5/GfK/xpD/C4b8ryrkf04h/1cK8e8oxL+bEP8uQvybCPFvHMS/cRD/dkH82wXxbxXEv1UQ/y5B/NsD8e8OxL8nEP9OQPy7APFvAkTZf1HuX5T1F2X4RZl8UQ5flL0X5exFmXlRXl6UcRdl1kWZdVFmXZRZF2XWRZl1UWZdlFkXZdZFmXVRZl2UWRdl1kWZdVFmXZRZF2XWRXl0Ud5clBsX5cZFOXBR7luU+xZlvEV5blH2WpRFFmWORRljUcZYlPkV5X1FGV1RLleUsxVla0VZWFH2VZRHFWVPRdlTUTZUlAkV5UFF2U9RtlOU2xTlLkVZS1HWUpSlFOUiRfk/UZZPlKsTZeg0nmiPv/4r3kHvkO8w32G9Q73DfofTjc3SPKOZRnONZhvNN5pxNOdo1tG80bzh5zVvNG80bzRvNG80bzRvNK+aV80rd7Z51bxqXjWvmlfNK/P6wS3aLrYnF9uTi+3JxfbkYntysT252J5cbE8utieK7Ylie6LYnii2J4rtiWJ7otieKLYniu2JYnui2J4otieK7Ylie6LYnii2J4rtiWJ7otieKLYniu2JYnui2J4otieK7Ylie6LYnii2J4rdiWJ3otidKHYnit2JYnei2J0odieK3Ylid6LYnSh2J4rdiWJ3otidKHYnit2JYnei2J0odieK3Ylid6LYnSh2J4rdiWJ3otidKHYnit2JYnei2J0odieK3Ylid6LYnSh2J4rdiWJ3otidKHYnit3Jxe7kYndysTu52J1c7E4udicXu5OL3cnF7uRid3KxO1HsThS7E8XuRLE7UexOFLsTxe5EsTtR7E4UuxPF7kSxO1HsThS7E8XuRLE7UexOFLsTxe5EsTtR7E4UuxPF7kSxO1HsThS7E8XuHn2mpCh6J4rOiaJzouicKDonis6JonOi6JwoOieKzomic6LonCg6Z7pNV7c3CCk+J4rPieJzovicKD4nis+J4nOi+JwoPieKz4nic6L4nCg+J4rPieJzovicKD4nis89+kw5U7dvUFKMThSjE8XoRDE6UYxOFKMTxehEMTrp6Xv99V/xDnqHfIf5Dusd6h32O5xubJbmGc00mms022i+0YyjOUezjuaN5g0/r3mjeaN5o3mjeeWmLSyujidXxxM17URNO1HLTtSyE7XrRK06UatO1KgTtelELTpRi07UoBM16EQNOlETTtSEEzXgRM03UetN1HoTtd5EbTdR203UdBM13URNN1HTTdRyEzXbRM02UbNN1GwTNdpETTZRk03UYtOrxXYP/Ve8g94h32G+w3qHeof9Dqcbm6V5RjON5hrNNppvNONoztGso3mjecPPa95o3mjeaN5o3mjeaN5oXjWvmlfubPOqedW8al41r5pX5vWDGyFphKQRkiAkQQj1a0Q9GlG/RdQVEfVBRJ2Nfqfpd2uEJAhhn7/Y1y/2tbsvpo0Q9n2K/ZFiH5DYxyP237zO/8xvhJBnL/LlRR64yOd+r9cIIf9S5E+KvEaRbyjy70T+ici3EHF3vbi7XtxdL+6uF3fXi7vrxd314u56cXe9uLte3F0dd1fH3dVxd3XcXR13V8fd1XF3ddxdHXdXx93VcXd13F0dd1fH3dVxd3XcXR13V8fd1XF3ddxdHXdXx93VcXd13F0dd1fH3dVxd3XcXR13V8fd5bi7HHeXA+5ywF2EwkUoXITARQhchLxFiFuEuEVoW4S0RQhbhLBF6FqErkXoWoSqRahahKZFKFqEmEWIWYSYRUhZhJRFKFmEkkUoWYSSRQhZhIpFqFiEikWoWISGRehXhH5FyFcv5KsX8tUL+eqFfPVCvnohX72Qr17IVy/kqxfyVYd81SFfdchXHfJVh3zVIV91yFcd8lWHfNUhX3XIVx3yVYd81SFfdchXHfJVh3zVIV91yFcd8lWHfNUhX3XIVx3yVYd81SFfdchXHfJVh3zVIV855CuHfOVYrxzrFRFaEaEVkVkRmRWRWBF5FZFXEXEVkVYRWRWRVRFRFRHVfqOf37jczkghkikilyJSKSKQptt0Nb8RQ+RRRB5FxFFEHEXEUUQcRaRRRBBFBFFEEEUE8dG2QBYWCBFEETEUEUMRKVQ9BNVf/xXvoHfId5jvsN6h3mG/w+nGZmme0UyjuUazjeYbzTiaczTraN5o3vDzmjeaN5o3mjeaN5o3mjeaV82r5pU727xqXjWvmlfNq+aVef3gRpADoXIgVIQvRfhShC1F2FKEKUVYUoQlRThShCFF2FGEHUW4UYQbRZhRhPVEWE+E8UTYToTrRJhOhOlEWE6E5UQ4ToTjRDhOhONEGE6E20S4TYTbRLhNhNNE+EyEz0TYTC9sphc20wub6YXN9MJmemEzvbCZXthML2ymFzZTh83UYTN12EwdNlOHzdRhM3XYTB02U4fN1GEzddhMHTZTh83UYTN12EwdNlOHzdRhM3XYTB02U4fN1GEzddhMHTZTh83UYTN12EwdNlOHzdRhMzlsJofN5HiZHC8TUS4R5RLRLRHdEtEsEb0S0SsRtRLRKhGdEtEpEZUSUal+o5/fuNzOSCEaJKI/Itojojym23Q1vxFDdEdEd0RUR0R1RFRHRHVENEdEbUTURkRtRNTm0dYxGx1D1EZEaUSURkRndB6Czl//Fe+gd8h3mO+w3qHeYb/D6cZmaZ7RTKO5RrON5hvNOJpzNOto3mje8POaN5o3mjeaN5o3mjeaN5pXzavmlTvbvGpeNa+aV82r5pV5/eBGkINPcvBJhIxEyEiEikSoSISGRChIhIJECEiEfkSoR4R6RIhHhHhEaEeEUkQoRYRORKhEhEhEaESERkQoRIRCRAhEhEBECESEQEToQ4Q4RIhDhDhEiEOEMETIQoQs5FDF/2UTBFM='
text = zlib.decompress(base64.b64decode(BLOB)).decode()
open('skip_tiles.txt', 'w').write(text + '\n')
print(len(text.splitlines()), 'already-screened tiles excluded')


In [ ]:
!python batch_screen.py --dry-run --layers 20-39 --neurons 0-8192 --skip-file skip_tiles.txt --out /tmp/dry.jsonl


In [ ]:
!python batch_screen.py --layers 20-39 --neurons 0-8192 --probe "Who are you?" --dose 4.0 --batch 32 --max-new 50 --max-seconds 20000 --dtype float16 --skip-file skip_tiles.txt --out screen_L20_39.jsonl


In [ ]:
!wc -l screen_L20_39.jsonl
import json, collections
rows = [json.loads(l) for l in open('screen_L20_39.jsonl')]
print('rows', len(rows))
print('unique tiles', len({(r['layer'], r['neuron']) for r in rows}))
print('by layer', dict(collections.Counter(r['layer'] for r in rows).most_common(6)))
print('empty answers', sum(1 for r in rows if not r['answer'].strip()))
print('batch sizes', dict(collections.Counter(r['batch_size'] for r in rows)))
